In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc

from hm3d_utils.statistics import calculate_lisi_scores

In [4]:
adata = sc.read_h5ad('/REDACTED-path/HypoMap3D-data/v0.9/9_objects/singlecell/hypomap3d_singlecell_control.h5ad')

In [5]:
invalid_mask = adata.obs['sex_per_sample_inferred'] == 'mixed'
adata = adata[~invalid_mask, :]
adata

View of AnnData object with n_obs × n_vars = 839213 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'sex_per_sample_inferred', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'L1_ID', 'L2_ID', 'L3_ID', 'L4_ID', 'L5_ID', 'L1_descr', 'L2_descr', 'L3_descr', 'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like', 'L4_abc_like', 'L5_abc_like'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    obsm: 'X_embed', 'X_spatial_umap', 'X_umap'

In [6]:
lisi_scores = calculate_lisi_scores(
    embedding=adata.obsm['X_embed'],
    labels=adata.obs['sex_per_sample_inferred'].cat.codes,
    perplexity=30,
    k_factor=3,
    scaled=True,
    use_gpu=True
)

In [7]:
slisi_scores = 1 - lisi_scores
slisi_scores

array([0.04032731, 0.86563575, 0.302994  , ..., 0.46072137, 0.45088327,
       0.15656984], shape=(839213,), dtype=float32)

In [8]:
adata.obs['slisi_sex'] = slisi_scores

/REDACTED-path/.temp/ipykernel_717313/4107870903.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['slisi_sex'] = slisi_scores


In [12]:
import pandas as pd
from natsort import natsorted

df = pd.concat(
    [
        adata.obs.groupby('L1_ID', observed=True)['slisi_sex'].quantile(0.75),
        adata.obs.groupby('L2_ID', observed=True)['slisi_sex'].quantile(0.75),
        adata.obs.groupby('L3_ID', observed=True)['slisi_sex'].quantile(0.75),
        adata.obs.groupby('L4_ID', observed=True)['slisi_sex'].quantile(0.75),
        adata.obs.groupby('L5_ID', observed=True)['slisi_sex'].quantile(0.75),
    ]
)

index = df.index.tolist()
sorted_index = natsorted(index, key=lambda x: x[2:])

df = df.loc[sorted_index]
df.index.name = 'cluster_id'

df.to_csv('/REDACTED-path/HypoMap3D-data/v0.9/publication/SupTable_05/celltype_slisi_sex.csv')